# 統計學在大數據中的應用

## 學習目標

完成本 Notebook 後，你將能夠：

1. 說明大數據情境下，平均數、標準差、p 值等傳統統計指標可能失真的原因。
2. 使用中位數、分位數、截尾平均等方法，降低長尾分佈與極端值的影響。
3. 以模擬資料觀察樣本數過大時，p 值可能過度敏感的現象。
4. 理解多重檢定造成偽陽性累積的風險，並使用 Bonferroni 與 FDR 方法進行修正。
5. 以行銷 A/B 測試情境整合效果量、信賴區間與實務意義，避免只看統計顯著性。


In [ ]:
# ── 環境設定 ────────────────────────────────────
# 載入本章節所需的 Python 套件，並設定亂數種子，讓每次執行都能得到相同的模擬結果。

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.multitest import multipletests

np.random.seed(42)
plt.rcParams['figure.figsize'] = (8, 4)
plt.rcParams['axes.grid'] = True

print('環境設定完成')


## 核心概念說明

在大數據分析中，資料量變大不代表統計推論一定更可靠。常見問題包括：

- **長尾分佈與偏態**：少數極端值可能大幅拉高平均數，使平均數不再能代表典型使用者。
- **離群值被稀釋**：異常事件比例很低時，整體平均或變異數可能幾乎沒有變化，導致異常難以被發現。
- **p 值膨脹**：樣本數非常大時，微小差異也可能得到很小的 p 值，但這不一定代表具有實務價值。
- **多重檢定**：同時檢查大量變數或群體時，即使所有差異都只是隨機誤差，也容易出現看似顯著的偽陽性。
- **非隨機樣本與非獨立觀測**：平台紀錄、點擊流、感測器資料常不是隨機抽樣，且同一使用者可能產生多筆相關紀錄。

因此，大數據中的統計分析不應只追求「顯著」，還要同時檢查資料代表性、效果量、信賴區間、分佈型態與實務意義。


In [ ]:
# ── 示範：長尾分佈下的敘述統計失真 ─────────────────────────
# 這段程式碼模擬消費金額資料。多數顧客消費金額不高，但少數高消費顧客會形成長尾分佈，進而拉高平均數。

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

np.random.seed(42)

# 模擬 100,000 筆消費金額：大多數人消費較低，少數人消費極高
regular_customers = np.random.lognormal(mean=3.2, sigma=0.45, size=99000)
vip_customers = np.random.lognormal(mean=7.2, sigma=0.7, size=1000)
amounts = np.concatenate([regular_customers, vip_customers])

desc = pd.Series(amounts).describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99])
# proportiontocut=0.05 代表「兩側各」截去 5%，保留中間 90% 的資料
trimmed_mean = stats.trim_mean(amounts, proportiontocut=0.05)

summary = pd.DataFrame({
    '指標': ['平均數', '中位數', '截尾平均（兩側各截 5%）', '第 90 百分位', '第 99 百分位'],
    '金額': [amounts.mean(), np.median(amounts), trimmed_mean, np.percentile(amounts, 90), np.percentile(amounts, 99)]
})

print(summary.round(2))

plt.hist(amounts, bins=80, color='steelblue', alpha=0.8)
plt.axvline(amounts.mean(), color='red', linestyle='--', label='平均數')
plt.axvline(np.median(amounts), color='green', linestyle='--', label='中位數')
plt.xlim(0, np.percentile(amounts, 99.5))
plt.title('長尾分佈：平均數容易被少數極端值拉高')
plt.xlabel('消費金額')
plt.ylabel('人數')
plt.legend()
plt.show()


## 樣本數過大時，p 值會失去鑑別力

樣本數越大，標準誤越小，於是再微小的差異也能得到很小的 p 值。以下比較兩個廣告版本的點擊率（0.1000 vs 0.1015），在兩百萬人的樣本下觀察 p 值、效果量與信賴區間，說明為什麼「顯著」不等於「有實務價值」。


In [ ]:
# ── 示範：樣本數過大造成 p 值過度敏感 ──────────────────────
# 這段程式碼比較兩個廣告版本的點擊率。即使差異只有 0.15 個百分點，在超大樣本下也可能得到很小的 p 值，因此必須同時檢查效果量與實務門檻。

import numpy as np
import pandas as pd
from scipy import stats

np.random.seed(42)

n_a = 2_000_000
n_b = 2_000_000
ctr_a = 0.1000
ctr_b = 0.1015

click_a = np.random.binomial(1, ctr_a, n_a)
click_b = np.random.binomial(1, ctr_b, n_b)

observed_a = click_a.mean()
observed_b = click_b.mean()
diff = observed_b - observed_a

# 兩比例 z 檢定
pooled = (click_a.sum() + click_b.sum()) / (n_a + n_b)
se = np.sqrt(pooled * (1 - pooled) * (1 / n_a + 1 / n_b))
z = diff / se
p_value = 2 * (1 - stats.norm.cdf(abs(z)))

# 95% 信賴區間
se_unpooled = np.sqrt(observed_a * (1 - observed_a) / n_a + observed_b * (1 - observed_b) / n_b)
ci_low = diff - 1.96 * se_unpooled
ci_high = diff + 1.96 * se_unpooled

practical_threshold = 0.002  # 例如公司設定至少提升 0.2 個百分點才值得上線

result = pd.DataFrame({
    '項目': ['A 點擊率', 'B 點擊率', '差異', 'p 值', '95% CI 下界', '95% CI 上界', '是否達實務門檻'],
    '數值': [observed_a, observed_b, diff, p_value, ci_low, ci_high, abs(diff) >= practical_threshold]
})

print(result)
print('\n判讀：p 值很小不代表一定值得採用，仍需檢查差異幅度是否足以支持商業決策。')


## 多重檢定與偽陽性

若一次只做一個檢定，顯著水準 α = 0.05 表示在虛無假設為真時，約有 5% 機率誤判為顯著。

但在大數據分析中，常會同時檢查數百或數千個特徵、族群、廣告版本或商品類別。若執行 1,000 次檢定，即使所有檢定實際上都沒有真實差異，也可能因隨機誤差產生約 50 個看似顯著的結果。

常見修正方式包括：

- **Bonferroni 修正**：將顯著門檻變得更嚴格，降低偽陽性，但也可能漏掉真實效果。
- **FDR 控制法**：控制被判定為顯著的結果中，預期有多少比例可能是偽發現，常用方法是 Benjamini-Hochberg 程序。

在實務上，修正方法要搭配問題風險選擇。例如醫療檢測通常更重視降低偽陽性與偽陰性，行銷實驗則可能更重視可行的商業收益與實驗成本。


In [ ]:
# ── 示範：多重檢定造成偽陽性 ────────────────────────────
# 這段程式碼模擬 1,000 個其實沒有差異的檢定。未修正時，仍可能出現多個 p 值小於 0.05 的假顯著結果；修正後可降低偽陽性風險。

import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests

np.random.seed(42)

num_tests = 1000
sample_size = 80
p_values = []

for _ in range(num_tests):
    group_a = np.random.normal(loc=0, scale=1, size=sample_size)
    group_b = np.random.normal(loc=0, scale=1, size=sample_size)
    _, p = stats.ttest_ind(group_a, group_b)
    p_values.append(p)

p_values = np.array(p_values)
raw_significant = (p_values < 0.05).sum()
bonf_reject, bonf_p, _, _ = multipletests(p_values, alpha=0.05, method='bonferroni')
fdr_reject, fdr_p, _, _ = multipletests(p_values, alpha=0.05, method='fdr_bh')

summary = pd.DataFrame({
    '方法': ['未修正 p < 0.05', 'Bonferroni 修正', 'FDR 修正'],
    '判定顯著數量': [raw_significant, bonf_reject.sum(), fdr_reject.sum()]
})

print(summary)
print('\n重點：大量檢定時，不能直接把所有 p < 0.05 的結果都視為真實效果。')


## 實際應用：行銷 A/B 測試決策表

把前面的觀念收成一張可判讀的表：同時輸出點擊率差異、p 值、信賴區間與是否達到實務門檻，讓決策不只看單一個 p 值。


In [ ]:
# ── 實際應用：行銷 A/B 測試決策表 ───────────────────────
# 這段程式碼建立一個較完整的 A/B 測試判讀流程，同時輸出點擊率差異、p 值、信賴區間與是否達到實務門檻。

import numpy as np
import pandas as pd
from scipy import stats

np.random.seed(7)

campaigns = pd.DataFrame({
    'version': ['A', 'B'],
    'users': [300_000, 300_000],
    'true_ctr': [0.082, 0.083]
})

campaigns['clicks'] = [
    np.random.binomial(users, ctr)
    for users, ctr in zip(campaigns['users'], campaigns['true_ctr'])
]
campaigns['observed_ctr'] = campaigns['clicks'] / campaigns['users']

n_a, n_b = campaigns.loc[0, 'users'], campaigns.loc[1, 'users']
x_a, x_b = campaigns.loc[0, 'clicks'], campaigns.loc[1, 'clicks']
p_a, p_b = campaigns.loc[0, 'observed_ctr'], campaigns.loc[1, 'observed_ctr']
diff = p_b - p_a
relative_lift = diff / p_a

# 兩比例 z 檢定
pooled_ctr = (x_a + x_b) / (n_a + n_b)
se_pooled = np.sqrt(pooled_ctr * (1 - pooled_ctr) * (1 / n_a + 1 / n_b))
z_score = diff / se_pooled
p_value = 2 * (1 - stats.norm.cdf(abs(z_score)))

# 差異的 95% 信賴區間
se_unpooled = np.sqrt(p_a * (1 - p_a) / n_a + p_b * (1 - p_b) / n_b)
ci_low = diff - 1.96 * se_unpooled
ci_high = diff + 1.96 * se_unpooled

alpha = 0.05
practical_threshold = 0.002  # 至少提升 0.2 個百分點才視為具實務意義
statistically_significant = p_value < alpha
practically_meaningful = diff >= practical_threshold
recommend_launch = statistically_significant and practically_meaningful and ci_low > 0

summary = pd.DataFrame({
    '指標': [
        'A 版使用者數',
        'B 版使用者數',
        'A 版點擊數',
        'B 版點擊數',
        'A 版點擊率',
        'B 版點擊率',
        '絕對差異',
        '相對提升',
        'z 值',
        'p 值',
        '95% CI 下界',
        '95% CI 上界',
        '統計上顯著',
        '達實務門檻',
        '建議上線'
    ],
    '數值': [
        n_a,
        n_b,
        x_a,
        x_b,
        p_a,
        p_b,
        diff,
        relative_lift,
        z_score,
        p_value,
        ci_low,
        ci_high,
        statistically_significant,
        practically_meaningful,
        recommend_launch
    ]
})

print(campaigns[['version', 'users', 'clicks', 'observed_ctr']])
print('\nA/B 測試決策表：')
print(summary)

if recommend_launch:
    print('\n決策：B 版同時達到統計顯著與實務門檻，可以考慮上線。')
elif statistically_significant:
    print('\n決策：B 版可能有統計顯著差異，但效果未達實務門檻，不建議只因 p 值小就上線。')
else:
    print('\n決策：目前沒有足夠證據支持 B 版優於 A 版。')
